# Filter Transfermarkt Data for Big 5 Leagues

Creates cleaner, filtered versions of Transfermarkt data:
- Only players from Premier League, Bundesliga, La Liga, Serie A, Ligue 1
- Only market values since 2020
- Only players worth ≥ €100k

**Run this BEFORE the market values notebook to get cleaner matching.**

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
# File paths - adjust these if needed
RAW_DIR = os.path.join('..', 'data', 'raw', 'player_scores')
PLAYERS_FILE = os.path.join(RAW_DIR, 'players.csv')
VALUES_FILE = os.path.join(RAW_DIR, 'player_valuations.csv')

OUTPUT_PLAYERS = os.path.join(RAW_DIR, 'players_big5_filtered.csv')
OUTPUT_VALUES = os.path.join(RAW_DIR, 'player_valuations_big5_filtered.csv')

# Big 5 league competition IDs in Transfermarkt
# You may need to verify these IDs match your data
BIG5_LEAGUE_IDS = [
    'GB1',  # Premier League
    'L1',   # Bundesliga
    'ES1',  # La Liga
    'IT1',  # Serie A
    'FR1',  # Ligue 1
]

# Minimum market value to include (100k EUR)
MIN_MARKET_VALUE = 100000

# Earliest date to include valuations
CUTOFF_DATE = '2020-01-01'

## 1. Load Original Data

In [3]:
print("Loading players...")
players = pd.read_csv(PLAYERS_FILE)
print(f"✓ Total players: {len(players):,}")

print("\nLoading valuations...")
valuations = pd.read_csv(VALUES_FILE)
valuations['date'] = pd.to_datetime(valuations['date'])
print(f"✓ Total valuation records: {len(valuations):,}")

print("\nSample players:")
display(players.head())

print("\nSample valuations:")
display(valuations.head())

Loading players...
✓ Total players: 32,601

Loading valuations...
✓ Total valuation records: 496,606

Sample players:


,player_id,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,country_of_citizenship,...,foot,height_in_cm,contract_expiration_date,agent_name,image_url,url,current_club_domestic_competition_id,current_club_name,market_value_in_eur,highest_market_value_in_eur
0,10,Miroslav,Klose,Miroslav Klose,2015,398,miroslav-klose,Poland,Opole,Germany,...,right,184.0,NaN,ASBW Sport Marketing,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/miroslav-klose...,IT1,Società Sportiva Lazio S.p.A.,1000000.0,30000000.0
1,26,Roman,Weidenfeller,Roman Weidenfeller,2017,16,roman-weidenfeller,Germany,Diez,Germany,...,left,190.0,NaN,Neubauer 13 GmbH,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/roman-weidenfe...,L1,Borussia Dortmund,750000.0,8000000.0
2,65,Dimitar,Berbatov,Dimitar Berbatov,2015,1091,dimitar-berbatov,Bulgaria,Blagoevgrad,Bulgaria,...,NaN,NaN,NaN,CSKA-AS-23 Ltd.,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/dimitar-berbat...,GR1,Panthessalonikios Athlitikos Omilos Konstantin...,1000000.0,34500000.0
3,77,NaN,Lúcio,Lúcio,2012,506,lucio,Brazil,Brasília,Brazil,...,NaN,NaN,NaN,NaN,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/lucio/profil/s...,IT1,Juventus Football Club,200000.0,24500000.0
4,80,Tom,Starke,Tom Starke,2017,27,tom-starke,East Germany (GDR),Freital,Germany,...,right,194.0,NaN,IFM,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/tom-starke/pro...,L1,FC Bayern München,100000.0,3000000.0



Sample valuations:


,player_id,date,market_value_in_eur,current_club_id,player_club_domestic_competition_id
0,405973,2000-01-20,150000,3057,BE1
1,342216,2001-07-20,100000,1241,SC1
2,3132,2003-12-09,400000,126,TR1
3,6893,2003-12-15,900000,984,GB1
4,10,2004-10-04,7000000,398,IT1


## 2. Filter Players by League

In [4]:
print(f"Filtering to Big 5 leagues: {BIG5_LEAGUE_IDS}\n")

big5_players = players[
    players['current_club_domestic_competition_id'].isin(BIG5_LEAGUE_IDS)
].copy()

print(f"Players in Big 5 leagues: {len(big5_players):,}")
print(f"Reduction: {len(players) - len(big5_players):,} players removed")

print("\nPlayers by league:")
display(big5_players['current_club_domestic_competition_id'].value_counts())

Filtering to Big 5 leagues: ['GB1', 'L1', 'ES1', 'IT1', 'FR1']

Players in Big 5 leagues: 11,698
Reduction: 20,903 players removed

Players by league:


current_club_domestic_competition_id
IT1    3178
FR1    2248
ES1    2217
GB1    2181
L1     1874
Name: count, dtype: int64

## 3. Filter by Market Value

In [5]:
print(f"Filtering to players with market value >= €{MIN_MARKET_VALUE:,}\n")

before = len(big5_players)
big5_players = big5_players[
    big5_players['market_value_in_eur'] >= MIN_MARKET_VALUE
].copy()

print(f"Players after value filter: {len(big5_players):,}")
print(f"Removed: {before - len(big5_players):,} low-value players")

print("\nMarket value distribution:")
value_bins = [100000, 500000, 1000000, 5000000, 10000000, 50000000, 1000000000]
value_labels = ['€100k-500k', '€500k-1M', '€1M-5M', '€5M-10M', '€10M-50M', '€50M+']
big5_players['value_bin'] = pd.cut(
    big5_players['market_value_in_eur'], 
    bins=value_bins, 
    labels=value_labels
)
display(big5_players['value_bin'].value_counts().sort_index())

big5_players = big5_players.drop(columns=['value_bin'])

Filtering to players with market value >= €100,000

Players after value filter: 9,927
Removed: 1,771 low-value players

Market value distribution:


value_bin
€100k-500k    4358
€500k-1M      1347
€1M-5M        1943
€5M-10M        549
€10M-50M       802
€50M+           86
Name: count, dtype: int64

## 4. Filter Valuations

In [6]:
# Get list of player IDs
big5_player_ids = set(big5_players['player_id'].values)

print(f"Filtering valuations to Big 5 players...\n")

# Filter to Big 5 players
big5_valuations = valuations[
    valuations['player_id'].isin(big5_player_ids)
].copy()
print(f"Valuations for Big 5 players: {len(big5_valuations):,}")

# Filter by date
print(f"Filtering valuations to dates >= {CUTOFF_DATE}\n")
big5_valuations = big5_valuations[
    big5_valuations['date'] >= CUTOFF_DATE
].copy()
print(f"Valuations since 2020: {len(big5_valuations):,}")

# Filter by minimum value
big5_valuations = big5_valuations[
    big5_valuations['market_value_in_eur'] >= MIN_MARKET_VALUE
].copy()
print(f"After removing valuations < €100k: {len(big5_valuations):,}")

Filtering valuations to Big 5 players...

Valuations for Big 5 players: 181,816
Filtering valuations to dates >= 2020-01-01

Valuations since 2020: 74,521
After removing valuations < €100k: 73,036


## 5. Final Cleanup

In [7]:
# Update player list to only those with valuations since 2020
print("Ensuring all players have at least one valuation since 2020...\n")

players_with_recent_values = set(big5_valuations['player_id'].unique())
before = len(big5_players)

big5_players = big5_players[
    big5_players['player_id'].isin(players_with_recent_values)
].copy()

print(f"Final player count: {len(big5_players):,}")
print(f"Removed: {before - len(big5_players):,} players without recent valuations")

Ensuring all players have at least one valuation since 2020...

Final player count: 8,692
Removed: 1,235 players without recent valuations


## 6. Save Filtered Data

In [8]:
print("Saving filtered data...\n")

big5_players.to_csv(OUTPUT_PLAYERS, index=False)
print(f"✓ Saved players to: {OUTPUT_PLAYERS}")

big5_valuations.to_csv(OUTPUT_VALUES, index=False)
print(f"✓ Saved valuations to: {OUTPUT_VALUES}")

Saving filtered data...

✓ Saved players to: ..\data\raw\player_scores\players_big5_filtered.csv
✓ Saved valuations to: ..\data\raw\player_scores\player_valuations_big5_filtered.csv


## 7. Summary Statistics

In [9]:
print("=" * 60)
print("SUMMARY")
print("=" * 60)

print(f"\nOriginal players:        {len(players):,}")
print(f"Filtered players:        {len(big5_players):,} ({len(big5_players)/len(players)*100:.1f}%)")

print(f"\nOriginal valuations:     {len(valuations):,}")
print(f"Filtered valuations:     {len(big5_valuations):,} ({len(big5_valuations)/len(valuations)*100:.1f}%)")

print(f"\nPlayers by league:")
for league in BIG5_LEAGUE_IDS:
    count = (big5_players['current_club_domestic_competition_id'] == league).sum()
    print(f"  {league}: {count:,}")

print("\n" + "=" * 60)
print("✓ COMPLETE")
print("=" * 60)
print("\nUse these filtered files in your market values notebook:")
print(f"  PLAYERS_FILE = '{OUTPUT_PLAYERS}'")
print(f"  VALUES_FILE = '{OUTPUT_VALUES}'")
print("=" * 60)

SUMMARY

Original players:        32,601
Filtered players:        8,692 (26.7%)

Original valuations:     496,606
Filtered valuations:     73,036 (14.7%)

Players by league:
  GB1: 1,693
  L1: 1,475
  ES1: 1,762
  IT1: 2,147
  FR1: 1,615

✓ COMPLETE

Use these filtered files in your market values notebook:
  PLAYERS_FILE = '..\data\raw\player_scores\players_big5_filtered.csv'
  VALUES_FILE = '..\data\raw\player_scores\player_valuations_big5_filtered.csv'


## 8. Preview Filtered Data

In [10]:
print("Sample filtered players:")
display(big5_players.head(10))

print("\nSample filtered valuations:")
display(big5_valuations.head(10))

Sample filtered players:


,player_id,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,country_of_citizenship,...,foot,height_in_cm,contract_expiration_date,agent_name,image_url,url,current_club_domestic_competition_id,current_club_name,market_value_in_eur,highest_market_value_in_eur
9,215,Roque,Santa Cruz,Roque Santa Cruz,2015,1084,roque-santa-cruz,Paraguay,Asunción,Paraguay,...,right,193.0,2023-12-31 00:00:00,NaN,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/roque-santa-cr...,ES1,Málaga CF,250000.0,12000000.0
21,532,Claudio,Pizarro,Claudio Pizarro,2019,86,claudio-pizarro,Peru,Callao,Peru,...,right,184.0,NaN,NaN,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/claudio-pizarr...,L1,Sportverein Werder Bremen von 1899,400000.0,12000000.0
76,1586,Christian,Schulz,Christian Schulz,2015,42,christian-schulz,Germany,Bassum,Germany,...,left,185.0,NaN,ARP Sportmarketing,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/christian-schu...,L1,Hannover 96,100000.0,4500000.0
97,2050,Nelson,Valdez,Nelson Valdez,2014,24,nelson-valdez,Paraguay,Caaguazú,Paraguay,...,right,179.0,NaN,GG11,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/nelson-valdez/...,L1,Eintracht Frankfurt Fußball AG,100000.0,5000000.0
111,2421,Michael,Rensing,Michael Rensing,2019,38,michael-rensing,Germany,Lingen,Germany,...,right,190.0,NaN,Sports360 GmbH,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/michael-rensin...,L1,Fortuna Düsseldorf,300000.0,4500000.0
122,2857,Eldin,Jakupovic,Eldin Jakupovic,2022,29,eldin-jakupovic,Jugoslawien (SFR),Kozarac,Switzerland,...,right,191.0,2023-12-31 00:00:00,NaN,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/eldin-jakupovi...,GB1,Everton Football Club,100000.0,1500000.0
123,2865,Stephan,Lichtsteiner,Stephan Lichtsteiner,2019,167,stephan-lichtsteiner,Switzerland,Adligenswil,Switzerland,...,right,182.0,NaN,NaN,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/stephan-lichts...,L1,Fußball-Club Augsburg 1907,800000.0,17000000.0
150,3159,Valerio,Di Cesare,Valerio Di Cesare,2018,130,valerio-di-cesare,Italy,Roma,Italy,...,right,187.0,2024-06-30 00:00:00,MM-Management,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/valerio-di-ces...,IT1,Parma Calcio 1913,200000.0,650000.0
179,3291,Gareth,Barry,Gareth Barry,2017,984,gareth-barry,England,Hastings,England,...,left,184.0,NaN,ppsports,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/gareth-barry/p...,GB1,West Bromwich Albion,325000.0,18500000.0
187,3332,Wayne,Rooney,Wayne Rooney,2017,985,wayne-rooney,England,Liverpool,England,...,right,176.0,NaN,Triple S Sports,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/wayne-rooney/p...,GB1,Manchester United Football Club,2000000.0,65000000.0



Sample filtered valuations:


,player_id,date,market_value_in_eur,current_club_id,player_club_domestic_competition_id
298973,5794,2020-01-01,500000,3368,ES1
298974,21972,2020-01-01,200000,607,IT1
298975,34409,2020-01-01,1000000,873,GB1
298980,240803,2020-01-01,450000,276,IT1
298982,443710,2020-01-01,200000,79,L1
298983,659813,2020-01-01,100000,15,L1
298986,42787,2020-01-02,125000,167,L1
298997,239570,2020-01-02,200000,276,IT1
299001,304144,2020-01-02,300000,1390,IT1
299003,309929,2020-01-02,300000,4083,IT1
